In [7]:
# Name: Shaza Adelina Binti Khairullah
# Student ID: SN0107857
# CISB5123 Text Analytics - Lab Assignment 2

import pandas as pd
import numpy as np
import nltk
import re

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.sentiment import SentimentIntensityAnalyzer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Download necessary NLTK resources
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('vader_lexicon')

#Data Preprocessing 
df = pd.read_csv("Reviews.csv")


df = df[['Text', 'Score']]
df.dropna(inplace=True)

# Define sentiment labels based on Score
def label_sentiment(score):
    if score <= 2:
        return 'negative'
    elif score == 3:
        return 'neutral'
    else:
        return 'positive'

df['Sentiment'] = df['Score'].apply(label_sentiment)

# Sampling 10,000 rows for efficiency 
df = df.sample(10000, random_state=42)

stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    words = word_tokenize(text)
    words = [word for word in words if word not in stop_words]
    return " ".join(words)

df['Cleaned_Text'] = df['Text'].apply(clean_text)

# Using TF-IDF to convert text to numerical features
tfidf_vectorizer = TfidfVectorizer(max_features=5000)
X = tfidf_vectorizer.fit_transform(df['Cleaned_Text'])
y = df['Sentiment']

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


#Machine Learning: Naive Bayes
nb_model = MultinomialNB()
nb_model.fit(X_train, y_train)
y_pred_nb = nb_model.predict(X_test)

print("--- Naive Bayes Performance ---")
print("Accuracy:", accuracy_score(y_test, y_pred_nb))
print(classification_report(y_test, y_pred_nb))

#Machine Learning: Logistic Regression
lr_model = LogisticRegression(max_iter=200)
lr_model.fit(X_train, y_train)
y_pred_lr = lr_model.predict(X_test)

print("\n--- Logistic Regression Performance ---")
print("Accuracy:", accuracy_score(y_test, y_pred_lr))
print(classification_report(y_test, y_pred_lr))

#Lexicon-based: VADER
sia = SentimentIntensityAnalyzer()

def vader_sentiment(text):
    score = sia.polarity_scores(text)['compound']
    if score >= 0.05:
        return 'positive'
    elif score <= -0.05:
        return 'negative'
    else:
        return 'neutral'

df['VADER_Prediction'] = df['Cleaned_Text'].apply(vader_sentiment)

print("\n--- VADER Lexicon Performance ---")
print("Accuracy:", accuracy_score(df['Sentiment'], df['VADER_Prediction']))
print(classification_report(df['Sentiment'], df['VADER_Prediction']))

# Export extracted data 
df.to_csv("processed_reviews.csv", index=False)



[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


--- Naive Bayes Performance ---
Accuracy: 0.782
              precision    recall  f1-score   support

    negative       0.93      0.05      0.09       298
     neutral       0.00      0.00      0.00       152
    positive       0.78      1.00      0.88      1550

    accuracy                           0.78      2000
   macro avg       0.57      0.35      0.32      2000
weighted avg       0.74      0.78      0.69      2000



C:\Users\User\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\User\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\User\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])



--- Logistic Regression Performance ---
Accuracy: 0.819
              precision    recall  f1-score   support

    negative       0.77      0.35      0.48       298
     neutral       0.33      0.02      0.04       152
    positive       0.82      0.99      0.90      1550

    accuracy                           0.82      2000
   macro avg       0.64      0.45      0.47      2000
weighted avg       0.78      0.82      0.77      2000


--- VADER Lexicon Performance ---
Accuracy: 0.7931
              precision    recall  f1-score   support

    negative       0.54      0.30      0.39      1398
     neutral       0.10      0.03      0.04       750
    positive       0.83      0.95      0.89      7852

    accuracy                           0.79     10000
   macro avg       0.49      0.43      0.44     10000
weighted avg       0.74      0.79      0.75     10000

